In [ ]:
from torchvision import datasets
from torchvision.transforms import v2
import torch
from torch import nn
from torchvision import models
from sklearn import model_selection
import matplotlib.pyplot as plt
import utils
import metrics
import tqdm

plt.style.use('default')

In [ ]:
COLORS = [
    "#FF0000",  # Red
    "#00FF00",  # Green
	"#0000FF",  # Blue
    "#FFFF00",  # Yellow
    "#FF00FF",  # Magenta
    "#00FFFF",  # Cyan
	"#FFA500",  # Orange
    "#B1636F",  # Purple
    "#BED944",
    "#9792D4FF",
]

SEED = 0

COLORED_PROPORTIONS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]	# Proportions of colored images in train and val images used for experimenting
TRAIN_BS = 32
VAL_BS = 32

In [ ]:
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
train_dataset = datasets.MNIST(root="./data", download=True)
test_dataset = datasets.MNIST(root="./data", train=False, download=True)

In [ ]:
print(train_dataset)
print(test_dataset)

In [ ]:
train_transforms = v2.Compose(
	[
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
	]
)
test_transforms = v2.Compose(
	[
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
	]
)

### Display proportions before train/val split

In [ ]:
labels = torch.tensor([y for x, y in train_dataset])

plt.hist(labels, bins=10)
plt.title("Train Dataset Label Distribution before splitting")

In [ ]:
train_split, val_split = model_selection.train_test_split(train_dataset, test_size=0.2, stratify=labels, random_state=SEED)

### Display proportions after train/val split

In [ ]:
train_labels = torch.tensor([y for x, y in train_split])
val_labels = torch.tensor([y for x, y in val_split])

plt.hist(train_labels, bins=10)
plt.title("Train Dataset Label Distribution")
plt.show()

plt.hist(val_labels, bins=10)
plt.title("Val Dataset Label Distribution")
plt.show()

### Convert to torch dataset

In [ ]:
class ColoredMNISTDataset(torch.utils.data.Dataset):
	"""Wrapper class that applies coloring based on the target class ID."""
	def __init__(self, dataset_list, transform=None, colored_proportions=0.95):
		"""
			Initialize 
		"""
		super().__init__()
		self.pil_to_tensor = v2.Compose(
			[
				v2.ToImage(),
				v2.ToDtype(torch.float32, scale=True)
			]
		)

		self.x = []
		for x, y in dataset_list:
			if torch.rand(1).item() < colored_proportions:
				colored_image = self._gray_to_colored(self.pil_to_tensor(x), y)
				self.x.append(colored_image)
			else:
				w, h = x.size
				self.x.append(self.pil_to_tensor(x).expand(1, 3, w, h))

		self.x = torch.cat(self.x, dim=0)
		self.y = torch.tensor([y for x, y in dataset_list])
		self.transform = transform

	def __len__(self):
		return len(self.x)
	
	def __getitem__(self, index):
		if self.transform is not None:
			return self.transform(self.x[index]), self.y[index]
		return self.x[index], self.y[index]
	
	def _gray_to_colored(self, x, y):
		assert len(COLORS) > y, f"Color index {y} is out of range for COLORS list."

		color = COLORS[y]
		colored_image = torch.zeros(3, x.shape[1], x.shape[2])
		colored_image[0] = x * int(color[1:3], 16) / 255.0
		colored_image[1] = x * int(color[3:5], 16) / 255.0
		colored_image[2] = x * int(color[5:7], 16) / 255.0
		return colored_image.unsqueeze(0)

In [ ]:
class Model(nn.Module):
	def __init__(self, n_classes=10):
		super().__init__()
		self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
		self.resnet.fc = nn.Linear(self.resnet.fc.in_features, n_classes)

	def forward(self, x):
		return self.resnet(x)		

In [ ]:
def train_one_epoch(model, train_dl, criterion, optimizer, metrics_calculator, device="cuda"):
	"""
	Performs the optimization over the whole dataset once.
	Resets the metrics_calculator at the beggining and updates metrics inplace but does not return the metrics directly.
	Those can be accesed outside of this function.

	Args:
		model: Model trained.
		train_dl: Dataloader with data used for training.
		criterion: Criterion used for calculating the loss.
		optimizer: Optimizer used for updating model's weights.
		metrics_calculator: Instance of metrics calculator used for computing metrics on train dataset.
		device (optional): Device used for calculations like CUDA or CPU.
	Returns:
		float: Mean training loss for current epoch.
	"""

	metrics_calculator.reset()
	model.train()
	train_epoch_loss = 0
	n_instances = 0
	
	for inputs, targets in tqdm.tqdm(train_dl, desc="Training"):
		optimizer.zero_grad()

		inputs = inputs.to(device)
		targets = targets.to(device)

		outputs = model(inputs)

		loss = criterion(outputs, targets)
		loss.backward()
		optimizer.step()

		train_epoch_loss += loss.item()
		metrics_calculator.update(outputs.detach(), targets.detach())
		n_instances += inputs.shape[0]
	
	mean_epoch_loss = train_epoch_loss / n_instances
	return mean_epoch_loss


def evaluate(model, dataloader, criterion, metrics_calculator, device="cuda"):
	"""
	Evaluates the model given a dataloader

	Args:
		model: Model to evaluate.
		dataloader: Dataloader with data used for training.
		criterion: Criterion used for calculating the loss.
		metrics_calculator: Instance of metrics calculator used for computing metrics.
		device (optional): Device used for calculations like CUDA or CPU.
	Returns:
		float: Mean loss.
	"""

	metrics_calculator.reset()
	model.eval()
	total_loss = 0
	n_instances = 0
	
	for inputs, targets in tqdm.tqdm(dataloader, desc="Evaluating"):
		inputs = inputs.to(device)
		targets = targets.to(device)

		outputs = model(inputs)

		loss = criterion(outputs, targets)

		total_loss += loss.item()
		metrics_calculator.update(outputs.detach(), targets.detach())
		n_instances += inputs.shape[0]
	
	mean_loss = total_loss / n_instances
	return mean_loss


def train(model, train_dl, val_dl, criterion, optimizer, metrics_calculator, epochs=100, device="cuda"):
	"""
	Trains the model for the specified number of epochs.

	Args:
		
	"""
	best_weights = model.state_dict()
	best_val_loss = float("inf")

	for epoch in range(epochs):
		train_loss = train_one_epoch(model, train_dl, criterion, optimizer, metrics_calculator, device)
		accuracy, precision, recall, f1_score, auprc, auroc = metrics_calculator.compute_all()
		val_loss = evaluate(model, val_dl, criterion, metrics_calculator, device)
		accuracy, precision, recall, f1_score, auprc, auroc = metrics_calculator.compute_all()

		print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}, AUPRC: {auprc:.4f}, AUROC: {auroc:.4f}")

In [ ]:
metrics_calculator = metrics.MetricsCalculator(num_classes=len(train_labels.unique()))

for proportion in COLORED_PROPORTIONS:
	metrics_calculator.reset()
	
	train_dataset = ColoredMNISTDataset(train_split, colored_proportions=proportion)
	val_dataset = ColoredMNISTDataset(val_split, colored_proportions=proportion)
	clean_val_dataset = ColoredMNISTDataset(val_split, colored_proportions=0)
	colored_val_dataset = ColoredMNISTDataset(val_split, colored_proportions=1)

	train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=TRAIN_BS, shuffle=True)
	val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=VAL_BS)
	clean_val_dataloader = torch.utils.data.DataLoader(clean_val_dataset, batch_size=VAL_BS)
	colored_val_dataloader = torch.utils.data.DataLoader(colored_val_dataset, batch_size=VAL_BS)

	model = Model()
	model.to(device)

	optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
	criterion = nn.CrossEntropyLoss()

	train(model, train_dataloader, val_dataloader, criterion, optimizer, metrics_calculator, device=device)

	print(f"Proportion of colored images: {proportion}")
	utils.show_from_dataset(train_dataset, idx=torch.arange(0, 10, 1).tolist())